# 28 — Custom Experiment Guide

This notebook demonstrates the new custom experiment capabilities added in the P0/P1/P2 patches:

1. **Custom sweep support** — `session.exp.custom()` with frequency, amplitude, and delay sweeps
2. **New experiment library methods** — 11 new `session.exp.*` methods
3. **@experiment decorator** — lightweight named experiment registration

**Prerequisites:** Run `00_hardware_definition.ipynb` first to initialize the session.

In [ ]:
# Standard qubox notebook imports
from qubox.notebook import *

## 1. Custom Experiments with Sweep Support

The `session.exp.custom()` method now supports sweep axes natively. When you provide a
`sweep=` parameter, the compiled QUA program automatically generates nested `for_()` loops
with the appropriate parameter updates.

### Supported sweep modes

| Parameter name | QUA behavior | QUA variable type |
|:---|:---|:---|
| `frequency`, `freq`, `if_frequency` | `update_frequency(target, var)` | `int` |
| `amplitude`, `gain`, `amp` | `play(op * amp(var), target)` | `fixed` |
| `duration`, `delay`, `wait` | `wait(var, target)` | `int` |

The sweep target is auto-inferred from the circuit gates. For explicit control, set
`metadata={"target": "qubit"}` on the `SweepAxis`.

### 1a. Frequency Sweep (Custom Spectroscopy)

In [ ]:
import numpy as np

# Define a custom spectroscopy sequence
seq = session.ops.sequence("custom_spec", [
    session.ops.play("saturation", "qubit"),
    session.ops.measure("readout"),
])

# Create a frequency sweep axis
freq_sweep = session.sweep.linspace(
    start=-50e6, stop=50e6, num=201,
    parameter="frequency",
    unit="Hz",
)

# Run — the QUA program will automatically contain:
#   with for_(*from_array(freq_var, frequencies)):
#       update_frequency("qubit", freq_var)
#       play("saturation", "qubit")
#       measure(..., "readout", ...)
result = session.exp.custom(
    sequence=seq,
    sweep=freq_sweep,
    n_avg=1000,
)

### 1b. Delay Sweep (Custom T1)

In [ ]:
# Define a T1 sequence with an unparameterized wait
# (no concrete duration → filled by sweep)
seq_t1 = session.ops.sequence("custom_t1", [
    session.ops.play("x180", "qubit"),
    session.ops.wait("qubit"),           # duration from sweep
    session.ops.measure("readout"),
])

# Delay sweep in clock cycles (1 clk = 4 ns)
delay_sweep = session.sweep.arange(
    start=4, stop=2500, step=10,
    parameter="delay",
    unit="clks",
)

result_t1 = session.exp.custom(
    sequence=seq_t1,
    sweep=delay_sweep,
    n_avg=500,
)

### 1c. Amplitude Sweep (Custom Power Rabi)

In [ ]:
# Amplitude sweep — gates without explicit amplitude use the sweep variable
seq_rabi = session.ops.sequence("custom_power_rabi", [
    session.ops.play("x180", "qubit"),   # amplitude from sweep
    session.ops.measure("readout"),
])

amp_sweep = session.sweep.linspace(
    start=0.0, stop=0.5, num=101,
    parameter="amplitude",
    unit="V",
)

result_rabi = session.exp.custom(
    sequence=seq_rabi,
    sweep=amp_sweep,
    n_avg=1000,
)

## 2. New Experiment Library Methods

The following methods have been added to `session.exp.*`:

| Sub-library | Method | Description |
|:---|:---|:---|
| `session.exp.qubit` | `.spectroscopy_ef()` | EF-transition spectroscopy |
| `session.exp.qubit` | `.sequential_rotations()` | Play named rotation sequence |
| `session.exp.qubit` | `.ramsey_chevron()` | 2D Ramsey (detuning × delay) |
| `session.exp.resonator` | `.spectroscopy_x180()` | Resonator spec with qubit excited |
| `session.exp.readout` | `.ge_raw_trace()` | Raw ADC trace for |g⟩ and |e⟩ |
| `session.exp.readout` | `.leakage_benchmark()` | Readout leakage characterization |
| `session.exp.reset` | `.passive_benchmark()` | Passive reset fidelity benchmark |
| `session.exp.storage` | `.ramsey()` | Storage cavity T2 |
| `session.exp.storage` | `.fock_spectroscopy()` | Fock-resolved spectroscopy |
| `session.exp.storage` | `.fock_ramsey()` | Fock-resolved Ramsey |
| `session.exp.storage` | `.fock_power_rabi()` | Fock-resolved power Rabi |

In [ ]:
# Example: EF-transition qubit spectroscopy
# Automatically prepares |e⟩ state, then sweeps drive frequency
result_ef = session.exp.qubit.spectroscopy_ef(
    qubit="qubit",
    readout="readout",
    freq=session.sweep.linspace(-50e6, 50e6, 201, parameter="frequency"),
    drive_amp=0.02,
    ge_prep_pulse="ge_x180",
    n_avg=1000,
)

In [ ]:
# Example: Resonator spectroscopy with qubit in |e⟩
result_x180 = session.exp.resonator.spectroscopy_x180(
    qubit="qubit",
    readout="readout",
    freq=session.sweep.linspace(-5e6, 5e6, 201, parameter="frequency"),
    r180="x180",
    n_avg=1000,
)

In [ ]:
# Example: 2D Ramsey chevron
result_chev = session.exp.qubit.ramsey_chevron(
    qubit="qubit",
    readout="readout",
    freq_span=10e6,
    df=100e3,
    max_delay=500,
    dt=4,
    n_avg=500,
)

## 3. @experiment Decorator

For commonly repeated custom experiments, use the `@experiment` decorator to register
them by name for easy reuse.

In [ ]:
from qubox.experiments import experiment, get_registered_experiments


@experiment("photon_number_ramsey", n_avg=500, category="storage")
def photon_number_ramsey(session, *, delay, disp_pulse="const_alpha"):
    """Custom n-resolved Ramsey combining displacement + selective pulse."""
    return session.exp.custom(
        sequence=session.ops.sequence("photon_ramsey", [
            session.ops.play(disp_pulse, "storage"),
            session.ops.play("sel_x90", "qubit"),
            session.ops.wait("qubit"),
            session.ops.play("sel_x90", "qubit"),
            session.ops.measure("readout"),
        ]),
        sweep=delay,
        n_avg=500,
    )


# View registered experiments
print(get_registered_experiments())

In [ ]:
# Run the registered experiment
delay_axis = session.sweep.arange(4, 2500, 10, parameter="delay", unit="clks")
result = photon_number_ramsey(session, delay=delay_axis)

## Summary

| Feature | API | Notes |
|:---|:---|:---|
| Custom sweeps | `session.exp.custom(sweep=axis)` | Auto-generates QUA for-loops |
| 31 registered experiments | `session.exp.<sub>.<method>()` | Up from 20 |
| @experiment decorator | `@experiment("name")` | Named experiment registry |
| Sweep-aware gates | implicit | Wait/amplitude gates use sweep QUA vars when unparameterized |